# Stage 2: Feature Engineering
Joins all cleaned tables, engineers date and financial features, one hot encodes categoricals, fills nulls, and splits 80/20.

Input: s3://ins-churn-data/cleaned/*.csv

Output: s3://ins-churn-data/features/features.csv, train.csv, val.csv, analytics_ready.csv, ml_ready.csv, feature_manifest.json

In [ ]:
import boto3, pandas as pd, numpy as np, io, os, json, logging
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

BUCKET_NAME = 'ins-churn-data'
REGION = 'us-east-1'
CLEAN_DIR = 'cleaned'
FEAT_DIR = 'features'
s3 = boto3.client('s3', region_name=REGION)


def read_csv_from_s3(bucket, key, **kwargs):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), **kwargs)


def write_csv_to_s3(df, bucket, key):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    log.info('  saved to s3://%s/%s (%d rows, %d cols)', bucket, key, len(df), df.shape[1])


def write_json_to_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key,
                  Body=json.dumps(data, indent=2, default=str))


log.info('Imports ready.')

In [ ]:
# Load cleaned tables
fact = read_csv_from_s3(BUCKET_NAME, f'{CLEAN_DIR}/fact_policy_activity_cleaned.csv')
customers = read_csv_from_s3(BUCKET_NAME, f'{CLEAN_DIR}/dim_customers_cleaned.csv')
products = read_csv_from_s3(BUCKET_NAME, f'{CLEAN_DIR}/dim_products_cleaned.csv')
agents = read_csv_from_s3(BUCKET_NAME, f'{CLEAN_DIR}/dim_agents_cleaned.csv')
geo = read_csv_from_s3(BUCKET_NAME, f'{CLEAN_DIR}/dim_geography_cleaned.csv')
bridge = read_csv_from_s3(BUCKET_NAME, f'{CLEAN_DIR}/bridge_customer_agent_cleaned.csv')

log.info('fact: %s', fact.shape)
log.info('customers: %s', customers.shape)
log.info('products: %s', products.shape)
log.info('agents: %s', agents.shape)
log.info('geo: %s', geo.shape)
log.info('bridge: %s', bridge.shape)

In [ ]:
# Safe join: drop overlapping columns from dims before merging.
# fact, customers, and geo all carry 'region', 'status', 'state' which would
# create _x/_y duplicates and break subsequent .isna().sum() scalar checks.

OVERLAP_CUSTOMERS = [c for c in ['region', 'status', 'state'] if c in customers.columns]
OVERLAP_GEO = [c for c in ['status', 'state'] if c in geo.columns]
OVERLAP_PRODUCTS = [c for c in ['status'] if c in products.columns]
OVERLAP_AGENTS = [c for c in ['status'] if c in agents.columns]

customers = customers.drop(columns=OVERLAP_CUSTOMERS, errors='ignore')
geo = geo.drop(columns=OVERLAP_GEO, errors='ignore')
products = products.drop(columns=OVERLAP_PRODUCTS, errors='ignore')
agents = agents.drop(columns=OVERLAP_AGENTS, errors='ignore')

# Rename customer PK so it aligns with fact's FK
if 'cust_id' in customers.columns:
    customers = customers.rename(columns={'cust_id': 'customer_id'})

df = fact.copy()
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(geo, on='location_id', how='left')
df = df.merge(agents, on='agent_id', how='left')

log.info('After joins: %s', df.shape)
assert not any('_x' in c or '_y' in c for c in df.columns), \
    f'Duplicate columns found: {[c for c in df.columns if "_x" in c or "_y" in c]}'

In [ ]:
# Date parsing and duration features
date_cols = ['policy_start_date', 'policy_end_date', 'event_date', 'join_date', 'hire_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if 'policy_end_date' in df.columns and 'policy_start_date' in df.columns:
    df['policy_duration_days'] = (df['policy_end_date'] - df['policy_start_date']).dt.days

if 'event_date' in df.columns and 'join_date' in df.columns:
    df['customer_tenure_days'] = (df['event_date'] - df['join_date']).dt.days

if 'event_date' in df.columns and 'hire_date' in df.columns:
    df['agent_tenure_days'] = (df['event_date'] - df['hire_date']).dt.days

# Drop raw date columns, not usable by XGBoost
df = df.drop(columns=[c for c in date_cols if c in df.columns], errors='ignore')
log.info('Date features engineered.')

In [ ]:
# Financial and margin features
if 'unit_price' in df.columns and 'cost' in df.columns:
    df['policy_margin'] = df['unit_price'] - df['cost']
    df['margin_pct'] = (df['policy_margin'] / df['unit_price'].replace(0, np.nan)).fillna(0)

log.info('Financial features engineered.')

In [ ]:
# One hot encode categoricals
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Keep analytics ready copy before encoding
analytics_ready = df.copy()
write_csv_to_s3(analytics_ready, BUCKET_NAME, f'{FEAT_DIR}/analytics_ready.csv')

df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
log.info('After one hot encoding: %s', df.shape)

In [ ]:
# Null handling
# Compute null counts once as a named Series to guarantee scalar indexing
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0].index.tolist()
log.info('%d columns have nulls, filling with median or 0', len(null_cols))

for col in null_cols:
    if df[col].dtype in [np.float64, np.int64, float, int]:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(0)

assert df.isnull().sum().sum() == 0, 'Nulls remain after fill'
log.info('No nulls remain.')

In [ ]:
# Save ML ready and full features
target_col = 'customer_lifetime_value'
feature_cols = [c for c in df.columns if c != target_col]

ml_ready = df[feature_cols].copy()  # features only, no target
write_csv_to_s3(ml_ready, BUCKET_NAME, f'{FEAT_DIR}/ml_ready.csv')
write_csv_to_s3(df, BUCKET_NAME, f'{FEAT_DIR}/features.csv')

# Train / val split (80/20)
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
write_csv_to_s3(df_train, BUCKET_NAME, f'{FEAT_DIR}/train.csv')
write_csv_to_s3(df_val, BUCKET_NAME, f'{FEAT_DIR}/val.csv')

# Feature manifest
manifest = {
    'total_features': len(feature_cols),
    'feature_names': feature_cols,
    'target': target_col,
    'train_rows': len(df_train),
    'val_rows': len(df_val),
}
write_json_to_s3(manifest, BUCKET_NAME, f'{FEAT_DIR}/feature_manifest.json')
log.info('Feature engineering complete.')